## Notebook Guide

This notebook prepares the dataset for model training.

### Responsibilities

1. Build the image index.
2. Create the reproducible stratified 70/15/15 split.
3. Save train/validation/test metadata as CSV files.
4. Verify image preprocessing.
5. Build the TensorFlow input pipeline.
6. Verify batches before model training.

### Important

This notebook does NOT train the CNN or MobileNet models.

The output of this notebook is the verified dataset/input pipeline used by
the model notebooks.

# Fresh vs Rotten Fruit Classification — Preprocessing

This notebook prepares the fruit image dataset for model training.

## Preprocessing goals

1. Build a labeled image index.
2. Create the required 70% training, 15% validation, and 15% testing split.
3. Preserve class proportions using stratified splitting.
4. Convert images to RGB.
5. Resize images to a common input size.
6. Normalize pixel values before model training.

In [ ]:
import kagglehub
from pathlib import Path

# Access the Kaggle dataset
dataset_path = kagglehub.dataset_download(
    "sriramr/fruits-fresh-and-rotten-for-classification"
)

DATASET_ROOT = Path(dataset_path) / "dataset"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

CLASS_NAMES = [
    "freshapples",
    "freshbanana",
    "freshoranges",
    "rottenapples",
    "rottenbanana",
    "rottenoranges",
]

NUM_CLASSES = len(CLASS_NAMES)

print("Dataset root:", DATASET_ROOT)
print("Number of classes:", NUM_CLASSES)
print("Classes:", CLASS_NAMES)

## 1. Build Image Index

In [ ]:
records = []

for split_name in ["train", "test"]:
    split_dir = DATASET_ROOT / split_name

    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name

        for image_path in class_dir.iterdir():
            if (
                image_path.is_file()
                and image_path.suffix.lower() in IMAGE_EXTENSIONS
            ):
                records.append(
                    {
                        "original_split": split_name,
                        "class_name": class_name,
                        "image_path": str(
                            image_path.relative_to(DATASET_ROOT)
                        ),
                    }
                )

print("Total image records:", len(records))

## 2. Create Image DataFrame

In [ ]:
import pandas as pd

df = pd.DataFrame(records)

print("DataFrame shape:", df.shape)
print("\nFirst 5 records:")
display(df.head())

In [ ]:
print("\nClass distribution:")
print(df["class_name"].value_counts())

print("\nOriginal split distribution:")
print(df["original_split"].value_counts())

## 3. Stratified Train / Validation / Test Split

The dataset is divided into 70% training, 15% validation, and 15% testing data.

Stratified splitting is used so that the class distribution is approximately preserved across all three subsets.

In [ ]:
from sklearn.model_selection import train_test_split

# First split:
# 70% training
# 30% temporary set
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["class_name"],
    random_state=42,
)

# Second split:
# 15% validation
# 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["class_name"],
    random_state=42,
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

print("Training images  :", len(train_df))
print("Validation images:", len(val_df))
print("Testing images   :", len(test_df))
print("Total images     :", len(train_df) + len(val_df) + len(test_df))

In [ ]:
print("Training distribution:")
print(train_df["class_name"].value_counts())

print("\nValidation distribution:")
print(val_df["class_name"].value_counts())

print("\nTesting distribution:")
print(test_df["class_name"].value_counts())

## 4. Save Dataset Splits

In [ ]:
from pathlib import Path

DATA_OUTPUT_DIR = Path("/content/fruit_split_metadata")
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_df[["image_path", "class_name"]].to_csv(
    DATA_OUTPUT_DIR / "train.csv",
    index=False
)

val_df[["image_path", "class_name"]].to_csv(
    DATA_OUTPUT_DIR / "validation.csv",
    index=False
)

test_df[["image_path", "class_name"]].to_csv(
    DATA_OUTPUT_DIR / "test.csv",
    index=False
)

print("Split files created:")
for file_path in sorted(DATA_OUTPUT_DIR.iterdir()):
    print(file_path.name, "→", file_path.stat().st_size, "bytes")

In [ ]:
print("Train CSV rows      :", len(train_df))
print("Validation CSV rows :", len(val_df))
print("Test CSV rows       :", len(test_df))

## 5. Load and Verify Saved Dataset Splits

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

train_split = pd.read_csv(DATA_DIR / "train.csv")
validation_split = pd.read_csv(DATA_DIR / "validation.csv")
test_split = pd.read_csv(DATA_DIR / "test.csv")

print("Training rows  :", len(train_split))
print("Validation rows:", len(validation_split))
print("Testing rows   :", len(test_split))
print("Total rows     :", len(train_split) + len(validation_split) + len(test_split))

print("\nColumns:")
print(train_split.columns.tolist())

print("\nFirst 5 training records:")
display(train_split.head())

Training rows  : 9519
Validation rows: 2040
Testing rows   : 2040
Total rows     : 13599

Columns:
['image_path', 'class_name']

First 5 training records:


,image_path,class_name
0,train/freshbanana/vertical_flip_Screen Shot 20...,freshbanana
1,test/rottenbanana/rotated_by_30_Screen Shot 20...,rottenbanana
2,train/rottenbanana/Screen Shot 2018-06-12 at 8...,rottenbanana
3,train/rottenapples/Screen Shot 2018-06-07 at 2...,rottenapples
4,train/rottenoranges/rotated_by_15_Screen Shot ...,rottenoranges


## 6. Image Preprocessing Configuration

Images will be converted to RGB and resized to 224 × 224 pixels.

The images will be normalized before being provided to the models.

In [ ]:
IMAGE_SIZE = (224, 224)
NUM_CHANNELS = 3

print("Target image size:", IMAGE_SIZE)
print("Number of channels:", NUM_CHANNELS)

## 7. Test Image Preprocessing

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Take the first image from the training split
sample_path = train_df.iloc[0]["image_path"]

full_sample_path = DATASET_ROOT / sample_path

with Image.open(full_sample_path) as image:
    image = image.convert("RGB")
    image = image.resize(IMAGE_SIZE)

    image_array = np.array(image, dtype=np.float32)
    normalized_image = image_array / 255.0

print("Image shape:", normalized_image.shape)
print("Minimum pixel value:", normalized_image.min())
print("Maximum pixel value:", normalized_image.max())

## 8. Reusable Image Preprocessing Function

In [ ]:
from PIL import Image
import numpy as np


def preprocess_image(image_path):
    """
    Load and preprocess one image.

    Steps:
    1. Open image
    2. Convert to RGB
    3. Resize to target size
    4. Convert to float32 NumPy array
    5. Normalize pixel values to 0-1
    """
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        image = image.resize(IMAGE_SIZE)

        image_array = np.array(image, dtype=np.float32)
        normalized_image = image_array / 255.0

    return normalized_image

In [ ]:
sample_path = DATASET_ROOT / train_df.iloc[0]["image_path"]

processed_image = preprocess_image(sample_path)

print("Shape:", processed_image.shape)
print("Data type:", processed_image.dtype)
print("Minimum:", processed_image.min())
print("Maximum:", processed_image.max())

## 9. Small Batch Preprocessing Test

In [ ]:
sample_paths = train_df["image_path"].head(6)

processed_images = []

for image_path in sample_paths:
    full_path = DATASET_ROOT / image_path
    processed_image = preprocess_image(full_path)
    processed_images.append(processed_image)

processed_images = np.array(processed_images)

print("Batch shape:", processed_images.shape)
print("Data type:", processed_images.dtype)
print("Minimum:", processed_images.min())
print("Maximum:", processed_images.max())

## 10. TensorFlow Training Input Pipeline

The preprocessed images will be provided to the model using TensorFlow's `tf.data` pipeline.

Images are loaded in batches instead of loading the complete dataset into memory.

In [ ]:
import tensorflow as tf
import numpy as np

# Convert class names into numeric labels.
CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

print("Class mapping:")
for class_name, index in CLASS_TO_INDEX.items():
    print(f"{class_name} -> {index}")


def load_and_preprocess_image(image_path, label):
    """
    Load one image and prepare it for the CNN.
    """
    image_bytes = tf.io.read_file(image_path)

    # Decode RGB images directly into 3 channels.
    image = tf.image.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    # Ensure TensorFlow knows the image has 3 channels.
    image.set_shape([None, None, 3])

    # Resize.
    image = tf.image.resize(image, IMAGE_SIZE)

    # Convert to float32 and normalize to 0-1.
    image = tf.cast(image, tf.float32) / 255.0

    return image, label


def create_dataset(dataframe, shuffle=False):
    """
    Create a TensorFlow dataset from image paths and class labels.
    """
    image_paths = [
        str(DATASET_ROOT / image_path)
        for image_path in dataframe["image_path"]
    ]

    labels = (
        dataframe["class_name"]
        .map(CLASS_TO_INDEX)
        .to_numpy(dtype=np.int32)
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(32)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [ ]:
train_ds = create_dataset(train_df, shuffle=True)
validation_ds = create_dataset(val_df, shuffle=False)
test_ds = create_dataset(test_df, shuffle=False)

print("Datasets created successfully.")

In [ ]:
for images, labels in train_ds.take(1):
    print("Batch image shape:", images.shape)
    print("Batch label shape:", labels.shape)
    print("Image dtype:", images.dtype)
    print("Minimum pixel value:", tf.reduce_min(images).numpy())
    print("Maximum pixel value:", tf.reduce_max(images).numpy())

## 11. Identify Original Image Sources

The dataset contains original images and pre-augmented versions of those
images.

To prevent data leakage, all augmented versions belonging to the same
original source image must be treated as one group.

A source identifier is created by removing the known augmentation prefix
from the filename and combining the resulting source filename with the
class name.

In [ ]:
import re
from pathlib import Path

In [ ]:
AUGMENTATION_PATTERNS = [
    r"^rotated_by_\d+_",
    r"^vertical_flip_",
    r"^translation_",
    r"^saltandpepper_",
]

In [ ]:
def extract_source_filename(image_path):
    """
    Remove known augmentation prefixes from an image filename
    to recover the original source filename.
    """
    filename = Path(image_path).name

    for pattern in AUGMENTATION_PATTERNS:
        filename = re.sub(pattern, "", filename, count=1)

    return filename

In [ ]:
print("Total image records:", len(df))
print("Unique source IDs:", df["source_id"].nunique())

print("\nSource group size distribution:")
print(df.groupby("source_id").size().value_counts().sort_index())

print("\nFirst 10 source IDs:")
print(
    df[
        ["image_path", "class_name", "source_filename", "source_id"]
    ].head(10)
)

## 12. Create a Leak-Free Group-Aware Split

The dataset is split at the original-source level rather than the individual
image level.

All augmented versions belonging to the same source image are kept in the
same split to prevent source-image leakage between training, validation,
and testing.

The split is stratified by class and uses a fixed random seed for
reproducibility.

Target split:

- 70% training
- 15% validation
- 15% testing

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Keep one row per original source group.
source_groups = (
    df[["source_id", "class_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Unique source groups:", len(source_groups))

# First split: 70% train, 30% temporary.
split_1 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_group_idx, temp_group_idx = next(
    split_1.split(
        source_groups["source_id"],
        source_groups["class_name"]
    )
)

train_source_ids = set(
    source_groups.iloc[train_group_idx]["source_id"]
)

temp_source_groups = source_groups.iloc[temp_group_idx].reset_index(drop=True)

# Second split: 50% validation, 50% test from the remaining 30%.
split_2 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_group_idx, test_group_idx = next(
    split_2.split(
        temp_source_groups["source_id"],
        temp_source_groups["class_name"]
    )
)

validation_source_ids = set(
    temp_source_groups.iloc[val_group_idx]["source_id"]
)

test_source_ids = set(
    temp_source_groups.iloc[test_group_idx]["source_id"]
)

# Assign each image row to the split of its source group.
df_leak_free = df.copy()

df_leak_free["split"] = df_leak_free["source_id"].apply(
    lambda source_id:
        "train" if source_id in train_source_ids
        else "validation" if source_id in validation_source_ids
        else "test"
)

print("\nSource-group counts:")
print(
    df_leak_free.groupby("split")["source_id"]
    .nunique()
    .sort_index()
)

print("\nImage-row counts:")
print(
    df_leak_free["split"].value_counts().sort_index()
)

print("\nClass distribution by split:")
print(
    pd.crosstab(
        df_leak_free["class_name"],
        df_leak_free["split"]
    )
)

## 13. Verify Zero Source Overlap

Before replacing the previous dataset splits, we verify that no original
source image appears in more than one split.

A source group must belong to exactly one of:

- Training
- Validation
- Testing

The assertions below must pass before the new split is accepted.

In [ ]:
train_sources = set(
    df_leak_free.loc[
        df_leak_free["split"] == "train",
        "source_id"
    ]
)

validation_sources = set(
    df_leak_free.loc[
        df_leak_free["split"] == "validation",
        "source_id"
    ]
)

test_sources = set(
    df_leak_free.loc[
        df_leak_free["split"] == "test",
        "source_id"
    ]
)

train_validation_overlap = train_sources & validation_sources
train_test_overlap = train_sources & test_sources
validation_test_overlap = validation_sources & test_sources

print("Train ↔ Validation overlap:", len(train_validation_overlap))
print("Train ↔ Test overlap:", len(train_test_overlap))
print("Validation ↔ Test overlap:", len(validation_test_overlap))

assert len(train_validation_overlap) == 0
assert len(train_test_overlap) == 0
assert len(validation_test_overlap) == 0

print("\nAll source-overlap checks passed.")